# Study of the latent representations

## *Experiences description*

### Experiment 3 - Autoencoders Trained on Increasing Climate Diversity

In this third experiment, we want to study the distribution shift of our data across climates inside an autoencoder (AE). So we focus on the latent representation of an AE trained on some climates.

We retrieve the patchs used in the second experiment. We randomly split these samples in train/val/test datasets for each climate.
We then train the AE on one of these configurations (using the train and validation sets) :
- historical climate
- historical and ssp245 climates
- historical, ssp245 and ssp370 climates
- all climates

Then we evaluate the reconstruction quality of the AE on all climates (using the test sets)

We then project the latent representations of the test sets into a common PCA space, and we plot some visualization of this PCA space. 

And finally we perform the same analyzes between distributions then in the second experiment : 
- compute multivariate shift metrics in that latent PCA space;
- analyze the moments of the latent principal components;
- analyze extreme latent scores;
- analyze seasonal shifts in latent PCA space.

### Experiment 4 - Invariant Autoencoder with Latent Alignment

In this fourth experiment, we dive a step closer to the real CERA architecture by adding an explicit climate invariance term to the autoencoder loss.

The idea is to test wether adding an alignment loss between climates will effectivelly bring different climate distributions closer compared to the raw data and to the simple AE architecture. Here we only train the AE on the historical climate and on SSP245 to reproduce the CERA architecture (one "cold" and one "warm" climate).

Training set:
- **historical + ssp245**

Loss:
- reconstruction loss on all samples;
- alignment loss between latent samples from **historical** and **ssp245**.

Loss equation : 
$$
L = L_{\text{rec}} + \lambda_{\text{Align}} \cdot \text{Align}(Z^{\text{hist}}_{\text{align}}, Z^{\text{ssp245}}_{\text{align}})
$$

Alignment method : 

Here we consider two different method to align the historical and SSP245 climates ;
- We consider a sliced Wasserstein alignment loss 
- And a adversarial classifier.

Note that in CERA, the method used is Earth Mover's Distance (EMD), but EMD can be expensive in high dimension, it is why we use a sliced Wasserstein alignment loss, which is a practical EMD-style approximation.

Interpretation:
- decreasing the alignment term should reduce latent distribution shift between historical and ssp245, and potentially between historical and other warmer climates;
- this must be balanced against reconstruction quality;
- PCA visualizations help determine whether the latent clouds become more mixed and allow us to calculate the same metrics as before onto our "normalized" PCA space.

### Experiment 5 - CERA-like architecture

In this fifth experiment, we add a predictor to the architecture considered in the fourth experiment. We thus now consider : AE (constructed either with cnn2D or MLPs) (and with either sliced wasserstein distance alignment or adversarial classifier alignment) and a predictor using only the aligned part of the historical climate latent representations. The predictor needs to predict the precipitation field (pr) over the whole grid of the samples. This architecture will be called a CERA-like architecture.

The same test/train/val split than before is used.

This CERA-like setup extends the invariant AE by adding a precipitation predictor:
- AE input: multivariate samples from historical + ssp245 climates.
- AE losses: reconstruction + latent alignment on the first 48 latent dimensions.
- Predictor: MLP on aligned latent dimensions (historical only) to predict `pr` over all 70 patch points.

Global loss used for AE update:
$$
L_{AE} = L_{rec} + \lambda_{align} L_{align} + \lambda_{pred} L_{pred}
$$

And we perform the predictor update at the same time, to be consistent with the end to end training used in the original CERA architecture.

## Part 0 - Global Configuration

**Library import**

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
import matplotlib.path as mpath
from matplotlib.patches import PathPatch
from scipy import stats
from scipy.stats import wasserstein_distance, ks_2samp
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from matplotlib.colors import to_rgba
from IPython.display import display
from cycler import cycler
import seaborn as sns
import json
from pathlib import Path
import pickle

**Style**

In [ ]:
# Reset to a clean baseline, then apply a global scientific style.
plt.style.use("default")
sns.set_theme(style="whitegrid", context="notebook", font="DejaVu Sans")

_plot_alpha = 0.8
_base_palette = [to_rgba(color, alpha=_plot_alpha) for color in ["#2F3B52", "#226592", "#CC7B39", "#A33124", "#6B7280", "#4B5563"]]


**Data Loading**

Chosen variables :

In [ ]:
num_sample = 1000000
chosen_autoencoder_type = "CNN" # choose between "MLP" and "CNN"
inv_alignment_method = "swd" # choose between "swd" and "adversarial"
variable = "pr" # variable to predict
val_fraction = 0.05
test_fraction = 0.15
cera_lambda_align = 0.0001
cera_lambda_pred = 0.01

In [ ]:
# Backward-compatible aliases used later in the notebook
climate_order = ["historical", "ssp126", "ssp245", "ssp370", "ssp585"]
climate_colors = {}
random_seed = 42

# Harmonized climate palette for all figures
preferred_climate_colors = {
    "historical": to_rgba("#2F3B52", alpha=_plot_alpha),  # dark navy-grey
    "ssp125": to_rgba("#1F77B4", alpha=_plot_alpha),      # light blue
    "ssp245": to_rgba("#2370A4", alpha=_plot_alpha),      # blue
    "ssp370": to_rgba("#F28E2B", alpha=_plot_alpha),      # orange
    "ssp585": to_rgba("#C0392B", alpha=_plot_alpha),      # deep red
}
for _climate_name, _color in preferred_climate_colors.items():
    if _climate_name in climate_order:
        climate_colors[_climate_name] = _color

In [ ]:
# Here we import the latent representations for the chosen variables

latent_root = Path("/glade/work/tsalin/CMIP/latent_representations")
if not latent_root.exists():
    raise FileNotFoundError(f"Latent representations directory not found: {latent_root}")

def _load_latent_payload(file_path):
    if not file_path.exists():
        raise FileNotFoundError(f"Latent representations file not found: {file_path}")

    with open(file_path, "rb") as handle:
        payload = pickle.load(handle)

    latent_by_climate = {}
    metadata_by_climate = {}
    for climate, entry in payload.items():
        if "latent" not in entry:
            raise KeyError(f"Missing 'latent' entry for climate '{climate}' in {file_path}")
        latent_by_climate[climate] = np.asarray(entry["latent"])
        metadata_by_climate[climate] = pd.DataFrame(entry["metadata"]).reset_index(drop=True)

    return latent_by_climate, metadata_by_climate


# ── exp3 ──────────────────────────────────────────────────────────────────────
_exp3_latent_dir = latent_root / "latent_representations_exp_3"

exp3_latent_file_AEh     = _exp3_latent_dir / f"exp3_ns{num_sample}_{chosen_autoencoder_type}_AEh_{variable}_{val_fraction}_{test_fraction}_latent_representations_df.pkl"
exp3_latent_file_AEhs2   = _exp3_latent_dir / f"exp3_ns{num_sample}_{chosen_autoencoder_type}_AEhs2_{variable}_{val_fraction}_{test_fraction}_latent_representations_df.pkl"
exp3_latent_file_AEhs2s3 = _exp3_latent_dir / f"exp3_ns{num_sample}_{chosen_autoencoder_type}_AEhs2s3_{variable}_{val_fraction}_{test_fraction}_latent_representations_df.pkl"
exp3_latent_file_AEall   = _exp3_latent_dir / f"exp3_ns{num_sample}_{chosen_autoencoder_type}_AEall_{variable}_{val_fraction}_{test_fraction}_latent_representations_df.pkl"

# ── exp4 ──────────────────────────────────────────────────────────────────────
exp4_latent_file = latent_root / "latent_representations_exp_4" / f"exp4_ns{num_sample}_{chosen_autoencoder_type}_{inv_alignment_method}_{variable}_{val_fraction}_{test_fraction}_{cera_lambda_align}_latent_representations_df.pkl"

# ── exp5 ──────────────────────────────────────────────────────────────────────
_exp5_latent_dir = latent_root / "latent_representations_exp_5"

exp5_cera_latent_file           = _exp5_latent_dir / f"cera_ns{num_sample}_{chosen_autoencoder_type}_{inv_alignment_method}_{variable}_{val_fraction}_{test_fraction}_{cera_lambda_align}_{cera_lambda_pred}_latent_representations_df.pkl"
exp5_cera_full_latent_file      = _exp5_latent_dir / f"cera_full_latent_ns{num_sample}_{chosen_autoencoder_type}_{inv_alignment_method}_{variable}_{val_fraction}_{test_fraction}_{cera_lambda_align}_{cera_lambda_pred}_latent_representations_df.pkl"
exp5_baseline2_latent_file      = _exp5_latent_dir / f"baseline_CERA_noalign_ns{num_sample}_{chosen_autoencoder_type}_{variable}_{val_fraction}_{test_fraction}_{cera_lambda_pred}_latent_representations_df.pkl"
exp5_baseline_climax_latent_file = _exp5_latent_dir / f"baseline_ClimaX_ns{num_sample}_{variable}_{val_fraction}_{test_fraction}_latent_representations_df.pkl"
exp5_cera_swdn_latent_file      = _exp5_latent_dir / f"cera_ns{num_sample}_{chosen_autoencoder_type}_swdn_{variable}_{val_fraction}_{test_fraction}_{cera_lambda_align}_{cera_lambda_pred}_latent_representations_df.pkl"

exp3_latent_test_by_climate_AEh,     exp3_latent_test_metadata_by_climate_AEh     = _load_latent_payload(exp3_latent_file_AEh)
exp3_latent_test_by_climate_AEhs2,   exp3_latent_test_metadata_by_climate_AEhs2   = _load_latent_payload(exp3_latent_file_AEhs2)
exp3_latent_test_by_climate_AEhs2s3, exp3_latent_test_metadata_by_climate_AEhs2s3 = _load_latent_payload(exp3_latent_file_AEhs2s3)
exp3_latent_test_by_climate_AEall,   exp3_latent_test_metadata_by_climate_AEall   = _load_latent_payload(exp3_latent_file_AEall)
exp4_latent_test_by_climate,         exp4_latent_test_metadata_by_climate         = _load_latent_payload(exp4_latent_file)
exp5_cera_latent_test_by_climate,         exp5_cera_latent_test_metadata_by_climate         = _load_latent_payload(exp5_cera_latent_file)
exp5_cera_full_latent_test_by_climate,    exp5_cera_full_latent_test_metadata_by_climate    = _load_latent_payload(exp5_cera_full_latent_file)
exp5_baseline2_latent_test_by_climate,    exp5_baseline2_latent_test_metadata_by_climate    = _load_latent_payload(exp5_baseline2_latent_file)
exp5_baseline_climax_latent_test_by_climate, exp5_baseline_climax_latent_test_metadata_by_climate = _load_latent_payload(exp5_baseline_climax_latent_file)
exp5_cera_swdn_latent_test_by_climate, exp5_cera_swdn_latent_test_metadata_by_climate = _load_latent_payload(exp5_cera_swdn_latent_file)

latent_test_sets = {
    "exp3_AEh": {
        "latent_test_by_climate": exp3_latent_test_by_climate_AEh,
        "latent_test_metadata_by_climate": exp3_latent_test_metadata_by_climate_AEh,
        "latent_file": exp3_latent_file_AEh,
    },
    "exp3_AEhs2": {
        "latent_test_by_climate": exp3_latent_test_by_climate_AEhs2,
        "latent_test_metadata_by_climate": exp3_latent_test_metadata_by_climate_AEhs2,
        "latent_file": exp3_latent_file_AEhs2,
    },
    "exp3_AEhs2s3": {
        "latent_test_by_climate": exp3_latent_test_by_climate_AEhs2s3,
        "latent_test_metadata_by_climate": exp3_latent_test_metadata_by_climate_AEhs2s3,
        "latent_file": exp3_latent_file_AEhs2s3,
    },
    "exp3_AEall": {
        "latent_test_by_climate": exp3_latent_test_by_climate_AEall,
        "latent_test_metadata_by_climate": exp3_latent_test_metadata_by_climate_AEall,
        "latent_file": exp3_latent_file_AEall,
    },
    "exp4": {
        "latent_test_by_climate": exp4_latent_test_by_climate,
        "latent_test_metadata_by_climate": exp4_latent_test_metadata_by_climate,
        "latent_file": exp4_latent_file,
    },
    "exp5_cera": {
        "latent_test_by_climate": exp5_cera_latent_test_by_climate,
        "latent_test_metadata_by_climate": exp5_cera_latent_test_metadata_by_climate,
        "latent_file": exp5_cera_latent_file,
    },
    "exp5_cera_full_latent": {
        "latent_test_by_climate": exp5_cera_full_latent_test_by_climate,
        "latent_test_metadata_by_climate": exp5_cera_full_latent_test_metadata_by_climate,
        "latent_file": exp5_cera_full_latent_file,
    },
    "exp5_baseline2": {
        "latent_test_by_climate": exp5_baseline2_latent_test_by_climate,
        "latent_test_metadata_by_climate": exp5_baseline2_latent_test_metadata_by_climate,
        "latent_file": exp5_baseline2_latent_file,
    },
    "exp5_baseline_climax": {
        "latent_test_by_climate": exp5_baseline_climax_latent_test_by_climate,
        "latent_test_metadata_by_climate": exp5_baseline_climax_latent_test_metadata_by_climate,
        "latent_file": exp5_baseline_climax_latent_file,
    },
    "exp5_cera_swdn": {
        "latent_test_by_climate": exp5_cera_swdn_latent_test_by_climate,
        "latent_test_metadata_by_climate": exp5_cera_swdn_latent_test_metadata_by_climate,
        "latent_file": exp5_cera_swdn_latent_file,
    }
}

print("Loaded latent representations:")
for latent_name, latent_payload in latent_test_sets.items():
    print(f"  {latent_name}: {latent_payload['latent_file']}")


## Part I - Common PCA of All Imported Latent Spaces

We build one common PCA space using all imported latent representations from:
- Experiment 3: AEh, AEhs2, AEhs2s3, AEall
- Experiment 4: full latent
- Experiment 5: cera full latent, baseline2 full latent, cera full_latent, baseline ClimaX full latent, and cera SWDN full latent

Then we normalize this common PCA space with its global centroid and global RMS dispersion, and visualize:
- explained variance;
- PC1/PC2 projection with both climate origin and latent-space origin highlighted.

We choose a PCA dimension target (used for this common PCA and for per-setup PCA computations later):

In [ ]:
num_PCA_components_for_ae = 15

Build the common PCA dataset from all imported full latent spaces and compute the normalized common PCA coordinates:

In [ ]:
# Common PCA across all imported full latent representations
from collections import OrderedDict

latent_source_by_name = OrderedDict({
    "AEh": (exp3_latent_test_by_climate_AEh, exp3_latent_test_metadata_by_climate_AEh),
    "AEhs2": (exp3_latent_test_by_climate_AEhs2, exp3_latent_test_metadata_by_climate_AEhs2),
    "AEhs2s3": (exp3_latent_test_by_climate_AEhs2s3, exp3_latent_test_metadata_by_climate_AEhs2s3),
    "AEall": (exp3_latent_test_by_climate_AEall, exp3_latent_test_metadata_by_climate_AEall),
    "exp4_full": (exp4_latent_test_by_climate, exp4_latent_test_metadata_by_climate),
    "exp5_cera_full": (exp5_cera_latent_test_by_climate, exp5_cera_latent_test_metadata_by_climate),
    "exp5_baseline2_full": (exp5_baseline2_latent_test_by_climate, exp5_baseline2_latent_test_metadata_by_climate),
    "exp5_cera_full_latent":   (exp5_cera_full_latent_test_by_climate,      exp5_cera_full_latent_test_metadata_by_climate),
    "exp5_baseline_climax_full": (exp5_baseline_climax_latent_test_by_climate, exp5_baseline_climax_latent_test_metadata_by_climate),
    "exp5_cera_swdn_latent_file": (exp5_cera_swdn_latent_test_by_climate, exp5_cera_swdn_latent_test_metadata_by_climate),

})

common_chunks = []
common_index = []
common_metadata_rows = []

for setup_name, (latent_by_climate, metadata_by_climate_setup) in latent_source_by_name.items():
    for climate in climate_order:
        Xc = np.asarray(latent_by_climate[climate])
        if Xc.ndim != 2:
            raise ValueError(f"Expected 2D latent array for {setup_name}/{climate}, got shape {Xc.shape}.")

        meta_c = metadata_by_climate_setup[climate].reset_index(drop=True)
        if len(meta_c) != Xc.shape[0]:
            raise ValueError(
                f"Metadata/sample size mismatch for {setup_name}/{climate}: "
                f"{len(meta_c)} metadata rows vs {Xc.shape[0]} samples."
            )

        common_chunks.append(Xc)
        common_index.append((setup_name, climate, Xc.shape[0]))

        season_values = meta_c["season"].values if "season" in meta_c.columns else np.array(["unknown"] * Xc.shape[0])
        common_metadata_rows.append(
            pd.DataFrame({
                "setup": setup_name,
                "scenario": climate,
                "season": season_values,
            })
        )

latent_common_matrix = np.vstack(common_chunks)
latent_common_metadata = pd.concat(common_metadata_rows, ignore_index=True)

latent_common_pca_components = min(
    num_PCA_components_for_ae,
    latent_common_matrix.shape[1],
    latent_common_matrix.shape[0],
)
latent_common_pca = PCA(n_components=latent_common_pca_components, random_state=random_seed)
latent_common_scores_all = latent_common_pca.fit_transform(latent_common_matrix)

# Normalize common latent PCA with global centroid and global RMS dispersion.
latent_common_global_centroid = np.mean(latent_common_scores_all, axis=0)
latent_common_scores_centered = latent_common_scores_all - latent_common_global_centroid
latent_common_global_rms_dispersion = float(
    np.sqrt(np.mean(np.sum(latent_common_scores_centered ** 2, axis=1)))
)
if latent_common_global_rms_dispersion <= 0:
    raise ValueError("Common latent PCA RMS dispersion is zero; normalization is undefined.")
latent_common_scores_all = latent_common_scores_centered / latent_common_global_rms_dispersion

latent_common_scores_df = pd.DataFrame(
    latent_common_scores_all,
    columns=[f"PC{i+1}" for i in range(latent_common_scores_all.shape[1])],
)
latent_common_scores_df = pd.concat([latent_common_scores_df, latent_common_metadata], axis=1)

# Split normalized common scores by setup and climate for downstream analysis.
latent_common_scores_by_setup_climate = {}
start = 0
for setup_name, climate, n_samples_setup_climate in common_index:
    stop = start + n_samples_setup_climate
    latent_common_scores_by_setup_climate.setdefault(setup_name, {})[climate] = latent_common_scores_all[start:stop]
    start = stop

print("Common latent PCA ready.")
print(f"Total samples used: {latent_common_matrix.shape[0]}")
print(f"Input latent dimension: {latent_common_matrix.shape[1]}")
print(f"PCA components used: {latent_common_pca_components}")
print(f"Global RMS dispersion (common PCA): {latent_common_global_rms_dispersion:.6g}")

Explained variance of the common latent PCA:

In [ ]:
# Explained variance for the common latent PCA
latent_common_explained = latent_common_pca.explained_variance_ratio_
latent_common_total_explained = float(np.sum(latent_common_explained))

print("Common latent PCA explained variance ratio:")
for i, ratio in enumerate(latent_common_explained, start=1):
    print(f"PC{i}: {ratio:.3%}")
print(f"Total explained variance: {latent_common_total_explained:.3%}")

fig, ax = plt.subplots(1, 1, figsize=(14, 5), constrained_layout=True)
ax.bar(
    np.arange(1, len(latent_common_explained) + 1),
    latent_common_explained * 100,
    color="#4c72b0",
    label="Explained variance",
)
ax.plot(
    np.arange(1, len(latent_common_explained) + 1),
    np.cumsum(latent_common_explained) * 100,
    marker="o",
    color="#dd8452",
    label="Cumulative explained variance",
)
ax.axhline(90, color="black", linestyle="--", linewidth=1.0, alpha=0.7)
ax.set_title("Common latent PCA explained variance")
ax.set_xlabel("Component")
ax.set_ylabel("Variance explained (%)")
ax.grid(alpha=0.25)
ax.legend()
plt.show()

PC1/PC2 representation with climate origin and latent-space origin both highlighted:

In [ ]:
from matplotlib.lines import Line2D

setup_marker_map = {
    "AEh": "o",
    "AEhs2": "s",
    "AEhs2s3": "^",
    "AEall": "D",
    "exp4_full": "P",
    "exp5_cera_full": "X",
    "exp5_baseline2_full": "v",
    "exp5_cera_full_latent":    "h",
    "exp5_baseline_climax_full": "*",
}

max_points_per_group_for_plot = 2500

fig, ax = plt.subplots(figsize=(12.5, 8), constrained_layout=True)

for setup_name in latent_source_by_name.keys():
    marker = setup_marker_map.get(setup_name, "o")
    setup_data = latent_common_scores_df[latent_common_scores_df["setup"] == setup_name]

    for climate in climate_order:
        group = setup_data[setup_data["scenario"] == climate]
        if group.empty:
            continue

        if len(group) > max_points_per_group_for_plot:
            group_plot = group.sample(n=max_points_per_group_for_plot, random_state=random_seed)
        else:
            group_plot = group

        ax.scatter(
            group_plot["PC1"],
            group_plot["PC2"],
            s=11,
            alpha=0.30,
            color=climate_colors[climate],
            marker=marker,
            linewidths=0,
        )

        cx = group["PC1"].mean()
        cy = group["PC2"].mean()
        ax.scatter(
            [cx],
            [cy],
            s=195,
            color=climate_colors[climate],
            marker=marker,
            edgecolor="black",
            linewidth=1.2,
            zorder=20,
        )

ax.set_title("Common latent PCA (PC1/PC2): climate colors + latent-space markers")
ax.set_xlabel("PC1")
ax.set_ylabel("PC2")
ax.grid(alpha=0.55)

climate_handles = [
    Line2D([0], [0], marker="o", color="w", markerfacecolor=climate_colors[c], markeredgecolor="black", markersize=8, label=c)
    for c in climate_order
]
setup_handles = [
    Line2D([0], [0], marker=setup_marker_map[s], color="black", linestyle="None", markersize=8, label=s)
    for s in latent_source_by_name.keys()
]

legend_climate = ax.legend(handles=climate_handles, title="Climate", loc="upper right")
ax.add_artist(legend_climate)
ax.legend(handles=setup_handles, title="Latent space", loc="lower left")

plt.show()

## Part II analysis of our PCA

In [ ]:

def _project_in_common_pca(X: np.ndarray) -> np.ndarray:
    Z = latent_common_pca.transform(X)
    return (Z - latent_common_global_centroid) / latent_common_global_rms_dispersion

def _build_masked_input(X: np.ndarray, slice_mode: str) -> np.ndarray:
    """Return ONLY the sliced dimensions without padding with zeros."""
    X = np.asarray(X)
    if X.ndim != 2:
        raise ValueError(f"Expected 2D latent array, got shape {X.shape}.")

    if slice_mode == "full":
        return X.copy()
    if slice_mode == "aligned48":
        if X.shape[1] < 48:
            raise ValueError(f"Cannot take first 48 dimensions from latent dimension {X.shape[1]}.")
        return X[:, :48]  # Return ONLY the 48 first dimensions
    if slice_mode == "non_aligned16":
        if X.shape[1] < 16:
            raise ValueError(f"Cannot take last 16 dimensions from latent dimension {X.shape[1]}.")
        return X[:, -16:]  # Return ONLY the 16 last dimensions

    raise ValueError(f"Unknown slice_mode: {slice_mode}")

def _pad_sliced_input_for_pca(X_sliced: np.ndarray, slice_mode: str, full_dim: int) -> np.ndarray:
    """Pad sliced data back to full dimensionality for PCA projection.
    
    For aligned48: pads the last 16 dimensions with the mean of those dimensions from historical data.
    For non_aligned16: pads the first 48 dimensions with the mean of those dimensions from historical data.
    """
    if X_sliced.shape[1] == full_dim:
        return X_sliced  # Already full dimension
    
    X_padded = np.zeros((X_sliced.shape[0], full_dim), dtype=X_sliced.dtype)
    
    if slice_mode == "aligned48":
        X_padded[:, :48] = X_sliced
        # Pad the last 16 dimensions with zeros (these represent the non-aligned part)
    elif slice_mode == "non_aligned16":
        X_padded[:, -16:] = X_sliced
        # Pad the first 48 dimensions with zeros (these represent the aligned part)
    
    return X_padded

In [ ]:
# Compute and store all requested metrics in the geometry of the common PCA (no additional PCA fit)
from collections import OrderedDict

setup_metric_specs = OrderedDict({
    "AEh": {"source": "AEh", "slice_mode": "full"},
    "AEhs2": {"source": "AEhs2", "slice_mode": "full"},
    "AEhs2s3": {"source": "AEhs2s3", "slice_mode": "full"},
    "AEall": {"source": "AEall", "slice_mode": "full"},
    "exp4_aligned48": {"source": "exp4_full", "slice_mode": "aligned48"},
    "exp4_non_aligned16": {"source": "exp4_full", "slice_mode": "non_aligned16"},
    "exp4_full": {"source": "exp4_full", "slice_mode": "full"},
    "exp5_cera_aligned48": {"source": "exp5_cera_full", "slice_mode": "aligned48"},
    "exp5_cera_non_aligned16": {"source": "exp5_cera_full", "slice_mode": "non_aligned16"},
    "exp5_cera_full": {"source": "exp5_cera_full", "slice_mode": "full"},
    "exp5_baseline2_full": {"source": "exp5_baseline2_full", "slice_mode": "full"},
    "exp5_cera_full_latent":    {"source": "exp5_cera_full_latent",    "slice_mode": "full"},
    "exp5_baseline_climax_full": {"source": "exp5_baseline_climax_full", "slice_mode": "full"},
})


def _scores_by_climate_from_common_pca(source_name: str, slice_mode: str) -> dict:
    latent_by_climate_source, _ = latent_source_by_name[source_name]
    scores_by_climate_local = {}
    full_dim = latent_common_pca.n_features_in_
    
    for climate in climate_order:
        X_raw = np.asarray(latent_by_climate_source[climate])
        X_sliced = _build_masked_input(X_raw, slice_mode)
        X_proj_input = _pad_sliced_input_for_pca(X_sliced, slice_mode, full_dim)
        scores_by_climate_local[climate] = _project_in_common_pca(X_proj_input)
    
    return scores_by_climate_local

def _per_component_ks_wasserstein(x_ref: np.ndarray, x_tgt: np.ndarray):
    ks_vals = []
    wass_vals = []
    n_components_local = x_ref.shape[1]
    for k in range(n_components_local):
        ks_vals.append(float(ks_2samp(x_ref[:, k], x_tgt[:, k]).statistic))
        wass_vals.append(float(wasserstein_distance(x_ref[:, k], x_tgt[:, k])))
    return np.array(ks_vals), np.array(wass_vals)

def _sliced_wasserstein_distance(x: np.ndarray, y: np.ndarray, n_projections: int = 128, seed: int = 42) -> float:
    rng = np.random.default_rng(seed)
    x = np.asarray(x, dtype=np.float64)
    y = np.asarray(y, dtype=np.float64)
    d = x.shape[1]
    projections = rng.normal(size=(n_projections, d))
    projections /= np.linalg.norm(projections, axis=1, keepdims=True) + 1e-12

    values = []
    for p in projections:
        xp = x @ p
        yp = y @ p
        values.append(float(wasserstein_distance(xp, yp)))
    return float(np.mean(values))

latent_setup_results = {}
pairwise_all_rows = []
moments_all_rows = []
moments_delta_all_rows = []
extremes_all_rows = []
extremes_delta_all_rows = []

for setup_name, spec in setup_metric_specs.items():
    source_name = spec["source"]
    slice_mode = spec["slice_mode"]
    setup_scores_by_climate = _scores_by_climate_from_common_pca(source_name, slice_mode)

    pairwise_rows = []
    for climate_i in climate_order:
        Xi = setup_scores_by_climate[climate_i]
        mu_i = np.mean(Xi, axis=0)
        for climate_j in climate_order:
            Xj = setup_scores_by_climate[climate_j]
            mu_j = np.mean(Xj, axis=0)

            if climate_i == climate_j:
                ks_vals = np.zeros(Xi.shape[1], dtype=float)
                wass_vals = np.zeros(Xi.shape[1], dtype=float)
                centroid_dist = 0.0
                swd = 0.0
            else:
                ks_vals, wass_vals = _per_component_ks_wasserstein(Xi, Xj)
                centroid_dist = float(np.linalg.norm(mu_i - mu_j))
                swd = _sliced_wasserstein_distance(Xi, Xj, n_projections=128, seed=random_seed)

            pairwise_rows.append({
                "setup": setup_name,
                "climate_i": climate_i,
                "climate_j": climate_j,
                "n_i": int(Xi.shape[0]),
                "n_j": int(Xj.shape[0]),
                "centroid_distance": centroid_dist,
                "ks_mean": float(np.mean(ks_vals)),
                "ks_max": float(np.max(ks_vals)),
                "wasserstein_mean": float(np.mean(wass_vals)),
                "wasserstein_max": float(np.max(wass_vals)),
                "sliced_wasserstein": float(swd),
            })

    pairwise_df = pd.DataFrame(pairwise_rows)
    distance_matrices = {
        metric: pairwise_df.pivot(index="climate_i", columns="climate_j", values=metric).loc[climate_order, climate_order]
        for metric in ["centroid_distance", "ks_mean", "wasserstein_mean", "sliced_wasserstein"]
    }

    moments_rows = []
    for climate in climate_order:
        Zc = setup_scores_by_climate[climate]
        z_flat = Zc.reshape(-1)
        moments_rows.append({
            "setup": setup_name,
            "scenario": climate,
            "mean": float(np.mean(z_flat)),
            "std": float(np.std(z_flat, ddof=0)),
            "skew": float(stats.skew(z_flat, bias=False, nan_policy="omit")),
            "kurtosis": float(stats.kurtosis(z_flat, fisher=True, bias=False, nan_policy="omit")),
        })

    moments_df = pd.DataFrame(moments_rows).set_index("scenario").loc[climate_order]
    hist_moments = moments_df.loc["historical", ["mean", "std", "skew", "kurtosis"]]
    moments_delta_df = moments_df.copy()
    for metric_name in ["mean", "std", "skew", "kurtosis"]:
        moments_delta_df[f"delta_{metric_name}_vs_historical"] = moments_df[metric_name] - hist_moments[metric_name]

    extremes_rows = []
    for climate in climate_order:
        Zc = setup_scores_by_climate[climate]
        norm = np.linalg.norm(Zc, axis=1)
        extremes_rows.append({
            "setup": setup_name,
            "scenario": climate,
            "q95_norm": float(np.quantile(norm, 0.95)),
            "q99_norm": float(np.quantile(norm, 0.99)),
            "max_norm": float(np.max(norm)),
        })

    extremes_df = pd.DataFrame(extremes_rows).set_index("scenario").loc[climate_order]
    hist_extremes = extremes_df.loc["historical", ["q95_norm", "q99_norm", "max_norm"]]
    extremes_delta_df = extremes_df.copy()
    for metric_name in ["q95_norm", "q99_norm", "max_norm"]:
        extremes_delta_df[f"delta_{metric_name}_vs_historical"] = extremes_df[metric_name] - hist_extremes[metric_name]

    latent_setup_results[setup_name] = {
        "spec": dict(spec),
        "common_pca_n_components": latent_common_pca_components,
        "common_pca_rms_dispersion": latent_common_global_rms_dispersion,
        "scores_by_climate": setup_scores_by_climate,
        "pairwise_metrics_df": pairwise_df,
        "distance_matrices": distance_matrices,
        "moments_df": moments_df,
        "moments_delta_df": moments_delta_df,
        "extremes_df": extremes_df,
        "extremes_delta_df": extremes_delta_df,
    }

    pairwise_all_rows.append(pairwise_df)
    moments_all_rows.append(moments_df.reset_index())
    moments_delta_all_rows.append(moments_delta_df.reset_index())
    extremes_all_rows.append(extremes_df.reset_index())
    extremes_delta_all_rows.append(extremes_delta_df.reset_index())

latent_pairwise_metrics_all_setups_df = pd.concat(pairwise_all_rows, ignore_index=True)
latent_moments_all_setups_df = pd.concat(moments_all_rows, ignore_index=True)
latent_moments_delta_all_setups_df = pd.concat(moments_delta_all_rows, ignore_index=True)
latent_extremes_all_setups_df = pd.concat(extremes_all_rows, ignore_index=True)
latent_extremes_delta_all_setups_df = pd.concat(extremes_delta_all_rows, ignore_index=True)

print("Finished computing and storing metrics for all requested setups.")
print("No setup-specific PCA was fitted: all metrics come from the common PCA geometry.")
print(f"Number of setups processed: {len(latent_setup_results)}")
print("Setups:")
for name in latent_setup_results.keys():
    print(f"  - {name}")

Quick check of the stored outputs:

In [ ]:
print("Stored aggregate tables:")
print(f"  pairwise rows: {len(latent_pairwise_metrics_all_setups_df)}")
print(f"  moments rows: {len(latent_moments_all_setups_df)}")
print(f"  moments delta rows: {len(latent_moments_delta_all_setups_df)}")
print(f"  extremes rows: {len(latent_extremes_all_setups_df)}")
print(f"  extremes delta rows: {len(latent_extremes_delta_all_setups_df)}")

## Part 3 - plotting the results

In [ ]:
# Centroid distance from historical to other climates, as a function of setup
target_climates = ["ssp245", "ssp370", "ssp585"]

plot_df = latent_pairwise_metrics_all_setups_df[
    latent_pairwise_metrics_all_setups_df["climate_i"] == "historical"
].copy()

# Keep setup order stable from the computation spec and put raw first when present
setup_order_base = list(setup_metric_specs.keys())
setup_values = set(plot_df["setup"].dropna().astype(str).unique())
if "raw" in setup_values:
    setup_order = ["raw"] + [s for s in setup_order_base if s != "raw"]
else:
    setup_order = setup_order_base

plot_df["setup"] = pd.Categorical(plot_df["setup"], categories=setup_order, ordered=True)
plot_df = plot_df.sort_values(["setup", "climate_j"])

fig, ax = plt.subplots(figsize=(12, 5), constrained_layout=True)

# Use the notebook base palette directly so the second-part figures stay aligned with the global theme.
pair_colors = dict(zip(target_climates, _base_palette[1:1 + len(target_climates)]))

for climate in target_climates:
    curve_df = plot_df[plot_df["climate_j"] == climate]
    if curve_df.empty:
        continue
    ax.plot(
        curve_df["setup"].astype(str),
        curve_df["centroid_distance"],
        marker="o",
        linewidth=2,
        color=pair_colors.get(climate),
        label=f"historical -> {climate}",
    )

ax.set_title("Centroid distance vs setup (from historical)")
ax.set_xlabel("Setup")
ax.set_ylabel("Centroid distance in common PCA space")
ax.grid(axis="y", alpha=0.25)
ax.legend(title="Climate pair")
plt.xticks(rotation=45, ha="right")

plt.show()

In [ ]:
# Alternative view: grouped bar chart for centroid distances from historical
target_climates = ["ssp245", "ssp370", "ssp585"]

plot_df = latent_pairwise_metrics_all_setups_df[
    latent_pairwise_metrics_all_setups_df["climate_i"] == "historical"
].copy()

# Keep setup order stable from the computation spec and put raw first when present
setup_order_base = list(setup_metric_specs.keys())
setup_values = set(plot_df["setup"].dropna().astype(str).unique())
if "raw" in setup_values:
    setup_order = ["raw"] + [s for s in setup_order_base if s != "raw"]
else:
    setup_order = setup_order_base

plot_df["setup"] = pd.Categorical(plot_df["setup"], categories=setup_order, ordered=True)
plot_df = plot_df.sort_values(["setup", "climate_j"]).reset_index(drop=True)

bar_df = (
    plot_df[plot_df["climate_j"].isin(target_climates)]
    .pivot(index="setup", columns="climate_j", values="centroid_distance")
    .reindex(setup_order)
    [target_climates]
    .fillna(np.nan)
).astype(float)

# Use the notebook base palette directly so the grouped bars match the line plot and the global theme.
bar_colors = [_base_palette[i + 1] for i in range(len(target_climates))]

fig, ax = plt.subplots(figsize=(13, 5.5), constrained_layout=True)
bar_df.plot(kind="bar", ax=ax, color=bar_colors, width=0.82)

ax.set_title("Centroid distance vs setup (grouped bars, from historical)")
ax.set_xlabel("Setup")
ax.set_ylabel("Centroid distance in common PCA space")
ax.grid(axis="y", alpha=0.25)

legend_handles = [
    Line2D([0], [0], marker="s", linestyle="None", markersize=10,
           markerfacecolor=color, markeredgecolor=color, label=f"historical -> {climate}")
    for climate, color in zip(target_climates, bar_colors)
]
ax.legend(handles=legend_handles, title="Climate pair")

plt.xticks(rotation=45, ha="right")

plt.show()

In [ ]:
# Ridgeline/Joyplot: PC1 distributions by (setup, climate)
target_climates = ["ssp245", "ssp370", "ssp585"]

if "latent_common_scores_by_setup_climate" not in globals():
    raise RuntimeError("latent_common_scores_by_setup_climate is missing. Run the setup/climate PCA cells first.")

# Desired setup list for this figure: remove full variants for exp4/exp5_cera/exp5_baseline2,
# and keep only aligned/non-aligned projections for those families.
target_setup_specs = [
    {"setup": "AEh", "mode": "precomputed"},
    {"setup": "AEhs2", "mode": "precomputed"},
    {"setup": "AEhs2s3", "mode": "precomputed"},
    {"setup": "AEall", "mode": "precomputed"},
    {"setup": "exp4_aligned48", "mode": "reproject", "source": "exp4_full", "slice_mode": "aligned48"},
    {"setup": "exp4_non_aligned16", "mode": "reproject", "source": "exp4_full", "slice_mode": "non_aligned16"},
    {"setup": "exp5_cera_aligned48", "mode": "reproject", "source": "exp5_cera_full", "slice_mode": "aligned48"},
    {"setup": "exp5_cera_non_aligned16", "mode": "reproject", "source": "exp5_cera_full", "slice_mode": "non_aligned16"},
    {"setup": "exp5_baseline2_aligned48", "mode": "reproject", "source": "exp5_baseline2_full", "slice_mode": "aligned48"},
    {"setup": "exp5_baseline2_non_aligned16", "mode": "reproject", "source": "exp5_baseline2_full", "slice_mode": "non_aligned16"},
    {"setup": "exp5_cera_full_latent", "mode": "reproject", "source": "exp5_cera_full_latent", "slice_mode": "full"},
    {"setup": "exp5_baseline_climax_full", "mode": "reproject", "source": "exp5_baseline_climax_full", "slice_mode": "full"},
]

rows = []
for spec in target_setup_specs:
    setup_name = spec["setup"]

    if spec["mode"] == "precomputed":
        setup_block = latent_common_scores_by_setup_climate.get(setup_name, {})
        for climate in target_climates:
            scores = setup_block.get(climate) if isinstance(setup_block, dict) else None
            if scores is None:
                continue

            arr = np.asarray(scores)
            if arr.size == 0:
                continue

            pc1 = arr[:, 0] if arr.ndim > 1 else arr.astype(float)
            pc1 = pc1[np.isfinite(pc1)]
            if pc1.size < 10:
                continue

            rows.append({"setup": setup_name, "climate": str(climate), "pc1": pc1})
        continue

    # Reproject requested latent slices in the already-fitted common PCA geometry.
    if spec["mode"] == "reproject":
        if "latent_source_by_name" not in globals() or "_build_masked_input" not in globals() or "_project_in_common_pca" not in globals() or "_pad_sliced_input_for_pca" not in globals():
            raise RuntimeError(
                "Missing latent source/projection helpers. Run the common PCA setup and metric cells first."
            )

        source_name = spec["source"]
        slice_mode = spec["slice_mode"]

        if source_name not in latent_source_by_name:
            continue

        latent_by_climate_source, _ = latent_source_by_name[source_name]
        full_dim = latent_common_pca.n_features_in_
        
        for climate in target_climates:
            if climate not in latent_by_climate_source:
                continue

            X_raw = np.asarray(latent_by_climate_source[climate])
            if X_raw.size == 0:
                continue

            X_sliced = _build_masked_input(X_raw, slice_mode)
            X_proj_input = _pad_sliced_input_for_pca(X_sliced, slice_mode, full_dim)
            scores = _project_in_common_pca(X_proj_input)
            arr = np.asarray(scores)
            if arr.size == 0:
                continue

            pc1 = arr[:, 0] if arr.ndim > 1 else arr.astype(float)
            pc1 = pc1[np.isfinite(pc1)]
            if pc1.size < 10:
                continue

            rows.append({"setup": setup_name, "climate": str(climate), "pc1": pc1})

if not rows:
    raise RuntimeError("No valid PC1 data found for the requested (setup, climate) groups.")

# Build x-domain from robust quantiles to keep a readable shared axis.
all_pc1 = np.concatenate([r["pc1"] for r in rows])
x_lo, x_hi = np.nanpercentile(all_pc1, [1.0, 99.0])
pad = 0.08 * (x_hi - x_lo + 1e-8)
x_grid = np.linspace(x_lo - pad, x_hi + pad, 500)

# Climate colors (fallback to notebook palette when unavailable).
default_climate_colors = {
    "ssp245": _base_palette[1],
    "ssp370": _base_palette[2],
    "ssp585": _base_palette[3],
}
if "preferred_climate_colors" in globals() and isinstance(preferred_climate_colors, dict):
    climate_palette = {k: preferred_climate_colors.get(k, default_climate_colors[k]) for k in target_climates}
elif "climate_colors" in globals() and isinstance(climate_colors, dict):
    climate_palette = {k: climate_colors.get(k, default_climate_colors[k]) for k in target_climates}
else:
    climate_palette = default_climate_colors

setup_order = [spec["setup"] for spec in target_setup_specs]
climate_rank = {c: i for i, c in enumerate(target_climates)}
rows = sorted(rows, key=lambda r: (setup_order.index(r["setup"]), climate_rank.get(r["climate"], 99)))

vertical_spacing = 0.31
ridge_height = 0.75

fig_h = max(8.0, 1.0 + vertical_spacing * len(rows))
fig, ax = plt.subplots(figsize=(12.8, fig_h), constrained_layout=True)

for idx, row in enumerate(rows):
    y0 = idx * vertical_spacing
    data = row["pc1"]

    kde = stats.gaussian_kde(data)
    dens = kde(x_grid)
    dens = dens / (dens.max() + 1e-12)
    ridge = y0 + ridge_height * dens

    color = climate_palette.get(row["climate"], _base_palette[4])
    ax.fill_between(x_grid, y0, ridge, color=color, alpha=0.42, linewidth=0)
    ax.plot(x_grid, ridge, color=color, linewidth=1.2, alpha=0.95)

    # Mean marker and robust spread endpoints (2%/98%).
    mu = float(np.mean(data))
    ql, qh = np.nanpercentile(data, [2.0, 98.0])

    ax.plot([mu, mu], [y0, y0 + 0.22], color="#1F2937", linewidth=1.1)
    ax.plot([ql, ql], [y0, y0 + 0.14], color="#4B5563", linewidth=0.9)
    ax.plot([qh, qh], [y0, y0 + 0.14], color="#4B5563", linewidth=0.9)

y_ticks = [i * vertical_spacing for i in range(len(rows))]
y_labels = [f"{r['setup']} | {r['climate']}" for r in rows]
ax.set_yticks(y_ticks)
ax.set_yticklabels(y_labels, fontsize=8)

ax.set_xlabel("PC1")
ax.set_ylabel("Setup | Climate")
ax.set_title("Ridgeline of PC1 distributions across setups and climates")
ax.grid(axis="x", alpha=0.22)
ax.grid(axis="y", alpha=0.06)

legend_handles = [
    Line2D([0], [0], color=climate_palette[c], linewidth=6, alpha=0.55, label=c)
    for c in target_climates
]
ax.legend(handles=legend_handles, title="Climate", loc="upper right")

# Tight vertical bounds so ridges look dense.
ax.set_ylim(-0.12, (len(rows) - 1) * vertical_spacing + ridge_height + 0.12)

plt.show()

In [ ]:
# Ridgeline/Joyplot: PC1 distributions by (setup, climate)
target_climates = ["historical", "ssp585"]

if "latent_common_scores_by_setup_climate" not in globals():
    raise RuntimeError("latent_common_scores_by_setup_climate is missing. Run the setup/climate PCA cells first.")

# Desired setup list for this figure: CERA / CERA SWDN / exp4 aligned & non-aligned slices,
# CERA full_latent, Baseline no-align (aligned slice only), Baseline ClimaX, and AE family setups.
target_setup_specs = [
    {"setup": "exp5_cera_aligned48", "mode": "reproject", "source": "exp5_cera_full", "slice_mode": "aligned48"},
    {"setup": "exp5_cera_non_aligned16", "mode": "reproject", "source": "exp5_cera_full", "slice_mode": "non_aligned16"},
    {"setup": "exp5_cera_full_latent", "mode": "reproject", "source": "exp5_cera_full_latent", "slice_mode": "full"},
    {"setup": "exp5_baseline2_aligned48", "mode": "reproject", "source": "exp5_baseline2_full", "slice_mode": "aligned48"},
    {"setup": "exp5_baseline_climax_full", "mode": "reproject", "source": "exp5_baseline_climax_full", "slice_mode": "full"},
    {"setup": "exp5_cera_swdn_aligned48", "mode": "reproject", "source": "exp5_cera_swdn_full", "slice_mode": "aligned48"},
    {"setup": "exp5_cera_swdn_non_aligned16", "mode": "reproject", "source": "exp5_cera_swdn_full", "slice_mode": "non_aligned16"},
    {"setup": "exp4_aligned48", "mode": "reproject", "source": "exp4_full", "slice_mode": "aligned48"},
    {"setup": "exp4_non_aligned16", "mode": "reproject", "source": "exp4_full", "slice_mode": "non_aligned16"},
    {"setup": "AEall", "mode": "precomputed"},
    {"setup": "AEhs2s3", "mode": "precomputed"},
    {"setup": "AEhs2", "mode": "precomputed"},
    {"setup": "AEh", "mode": "precomputed"},
]

rows = []
for spec in target_setup_specs:
    setup_name = spec["setup"]

    if spec["mode"] == "precomputed":
        setup_block = latent_common_scores_by_setup_climate.get(setup_name, {})
        for climate in target_climates:
            scores = setup_block.get(climate) if isinstance(setup_block, dict) else None
            if scores is None:
                continue

            arr = np.asarray(scores)
            if arr.size == 0:
                continue

            pc1 = arr[:, 0] if arr.ndim > 1 else arr.astype(float)
            pc1 = pc1[np.isfinite(pc1)]
            if pc1.size < 10:
                continue

            rows.append({"setup": setup_name, "climate": str(climate), "pc1": pc1})
        continue

    # Reproject requested latent slices in the already-fitted common PCA geometry.
    if spec["mode"] == "reproject":
        if "latent_source_by_name" not in globals() or "_build_masked_input" not in globals() or "_project_in_common_pca" not in globals() or "_pad_sliced_input_for_pca" not in globals():
            raise RuntimeError(
                "Missing latent source/projection helpers. Run the common PCA setup and metric cells first."
            )

        source_name = spec["source"]
        slice_mode = spec["slice_mode"]

        if source_name not in latent_source_by_name:
            continue

        latent_by_climate_source, _ = latent_source_by_name[source_name]
        full_dim = latent_common_pca.n_features_in_
        
        for climate in target_climates:
            if climate not in latent_by_climate_source:
                continue

            X_raw = np.asarray(latent_by_climate_source[climate])
            if X_raw.size == 0:
                continue

            X_sliced = _build_masked_input(X_raw, slice_mode)
            X_proj_input = _pad_sliced_input_for_pca(X_sliced, slice_mode, full_dim)
            scores = _project_in_common_pca(X_proj_input)
            arr = np.asarray(scores)
            if arr.size == 0:
                continue

            pc1 = arr[:, 0] if arr.ndim > 1 else arr.astype(float)
            pc1 = pc1[np.isfinite(pc1)]
            if pc1.size < 10:
                continue

            rows.append({"setup": setup_name, "climate": str(climate), "pc1": pc1})

if not rows:
    raise RuntimeError("No valid PC1 data found for the requested (setup, climate) groups.")

# Build x-domain from robust quantiles to keep a readable shared axis.
all_pc1 = np.concatenate([r["pc1"] for r in rows])
x_lo, x_hi = np.nanpercentile(all_pc1, [1.0, 99.0])
pad = 0.08 * (x_hi - x_lo + 1e-8)
x_grid = np.linspace(x_lo - pad, x_hi + pad, 500)

# Climate colors (fallback to notebook palette when unavailable).
default_climate_colors = {
    "historical": _base_palette[0],
    "ssp585": _base_palette[3],
}
if "preferred_climate_colors" in globals() and isinstance(preferred_climate_colors, dict):
    climate_palette = {k: preferred_climate_colors.get(k, default_climate_colors[k]) for k in target_climates}
elif "climate_colors" in globals() and isinstance(climate_colors, dict):
    climate_palette = {k: climate_colors.get(k, default_climate_colors[k]) for k in target_climates}
else:
    climate_palette = default_climate_colors

setup_order = [spec["setup"] for spec in target_setup_specs]
climate_rank = {c: i for i, c in enumerate(target_climates)}
rows = sorted(rows, key=lambda r: (setup_order.index(r["setup"]), climate_rank.get(r["climate"], 99)))

vertical_spacing = 0.31
ridge_height = 0.75

fig_h = max(8.0, 1.0 + vertical_spacing * len(rows))
fig, ax = plt.subplots(figsize=(12.8, fig_h), constrained_layout=True)

for idx, row in enumerate(rows):
    y0 = idx * vertical_spacing
    data = row["pc1"]

    kde = stats.gaussian_kde(data)
    dens = kde(x_grid)
    dens = dens / (dens.max() + 1e-12)
    ridge = y0 + ridge_height * dens

    color = climate_palette.get(row["climate"], _base_palette[4])
    ax.fill_between(x_grid, y0, ridge, color=color, alpha=0.42, linewidth=0)
    ax.plot(x_grid, ridge, color=color, linewidth=1.2, alpha=0.95)

    # Mean marker and robust spread endpoints (2%/98%).
    mu = float(np.mean(data))
    ql, qh = np.nanpercentile(data, [2.0, 98.0])

    ax.plot([mu, mu], [y0, y0 + 0.22], color="#1F2937", linewidth=1.1)
    ax.plot([ql, ql], [y0, y0 + 0.14], color="#4B5563", linewidth=0.9)
    ax.plot([qh, qh], [y0, y0 + 0.14], color="#4B5563", linewidth=0.9)

y_ticks = [i * vertical_spacing for i in range(len(rows))]
y_labels = [f"{r['setup']} | {r['climate']}" for r in rows]
ax.set_yticks(y_ticks)
ax.set_yticklabels(y_labels, fontsize=8)

ax.set_xlabel("PC1")
ax.set_ylabel("Setup | Climate")
ax.set_title("Ridgeline of PC1 distributions across setups and climates")
ax.grid(axis="x", alpha=0.22)
ax.grid(axis="y", alpha=0.06)

legend_handles = [
    Line2D([0], [0], color=climate_palette[c], linewidth=6, alpha=0.55, label=c)
    for c in target_climates
]
ax.legend(handles=legend_handles, title="Climate", loc="upper right")

# Tight vertical bounds so ridges look dense.
ax.set_ylim(-0.12, (len(rows) - 1) * vertical_spacing + ridge_height + 0.12)

plt.show()


In [ ]:
# Compact ridgeline: PC1 densities for selected setups and climates only
target_climates = ["historical", "ssp585"]

if "latent_common_scores_by_setup_climate" not in globals():
    raise RuntimeError("latent_common_scores_by_setup_climate is missing. Run the setup/climate PCA cells first.")

setup_specs = [
    {"label": "AEh", "mode": "precomputed", "setup": "AEh"},
    {"label": "AEhs2", "mode": "precomputed", "setup": "AEhs2"},
    {"label": "AEhs2s3", "mode": "precomputed", "setup": "AEhs2s3"},
    {"label": "AEall", "mode": "precomputed", "setup": "AEall"},
    {"label": "exp4_aligned", "mode": "reproject", "source": "exp4_full", "slice_mode": "aligned48"},
    {"label": "exp4_non_aligned", "mode": "reproject", "source": "exp4_full", "slice_mode": "non_aligned16"},
    {"label": "exp5_cera_aligned", "mode": "reproject", "source": "exp5_cera_full", "slice_mode": "aligned48"},
    {"label": "exp5_cera_non_aligned", "mode": "reproject", "source": "exp5_cera_full", "slice_mode": "non_aligned16"},
    {"label": "exp5_baseline2_aligned", "mode": "reproject", "source": "exp5_baseline2_full", "slice_mode": "aligned48"},
    {"label": "exp5_baseline2_non_aligned", "mode": "reproject", "source": "exp5_baseline2_full", "slice_mode": "non_aligned16"},
    {"label": "exp5_cera_latent",    "mode": "reproject", "source": "exp5_cera_full_latent",    "slice_mode": "full"},
    {"label": "exp5_baseline_climax","mode": "reproject", "source": "exp5_baseline_climax_full","slice_mode": "full"}
]

rows = []
for spec in setup_specs:
    if spec["mode"] == "precomputed":
        setup_block = latent_common_scores_by_setup_climate.get(spec["setup"], {})
        for climate in target_climates:
            scores = setup_block.get(climate) if isinstance(setup_block, dict) else None
            if scores is None:
                continue

            arr = np.asarray(scores)
            if arr.size == 0:
                continue

            pc1 = arr[:, 0] if arr.ndim > 1 else arr.astype(float)
            pc1 = pc1[np.isfinite(pc1)]
            if pc1.size < 10:
                continue

            rows.append({"setup": spec["label"], "climate": climate, "pc1": pc1})
        continue

    if "latent_source_by_name" not in globals() or "_build_masked_input" not in globals() or "_project_in_common_pca" not in globals() or "_pad_sliced_input_for_pca" not in globals():
        raise RuntimeError("Missing latent projection helpers. Run the setup metric computation cell first.")

    source_name = spec["source"]
    if source_name not in latent_source_by_name:
        continue

    latent_by_climate_source, _ = latent_source_by_name[source_name]
    full_dim = latent_common_pca.n_features_in_
    
    for climate in target_climates:
        if climate not in latent_by_climate_source:
            continue

        X_raw = np.asarray(latent_by_climate_source[climate])
        if X_raw.size == 0:
            continue

        X_sliced = _build_masked_input(X_raw, spec["slice_mode"])
        X_proj_input = _pad_sliced_input_for_pca(X_sliced, spec["slice_mode"], full_dim)
        scores = _project_in_common_pca(X_proj_input)

        arr = np.asarray(scores)
        pc1 = arr[:, 0] if arr.ndim > 1 else arr.astype(float)
        pc1 = pc1[np.isfinite(pc1)]
        if pc1.size < 10:
            continue

        rows.append({"setup": spec["label"], "climate": climate, "pc1": pc1})

if not rows:
    raise RuntimeError("No valid PC1 data found for the selected setups/climates.")

all_pc1 = np.concatenate([r["pc1"] for r in rows])
x_lo, x_hi = np.nanpercentile(all_pc1, [1.0, 99.0])
pad = 0.07 * (x_hi - x_lo + 1e-8)
x_grid = np.linspace(x_lo - pad, x_hi + pad, 420)

default_climate_colors = {
    "historical": _base_palette[0],
    "ssp585": _base_palette[3],
}
if "preferred_climate_colors" in globals() and isinstance(preferred_climate_colors, dict):
    climate_palette = {k: preferred_climate_colors.get(k, default_climate_colors[k]) for k in target_climates}
elif "climate_colors" in globals() and isinstance(climate_colors, dict):
    climate_palette = {k: climate_colors.get(k, default_climate_colors[k]) for k in target_climates}
else:
    climate_palette = default_climate_colors

setup_order = [s["label"] for s in setup_specs]
climate_rank = {c: i for i, c in enumerate(target_climates)}
rows = sorted(rows, key=lambda r: (setup_order.index(r["setup"]), climate_rank[r["climate"]]))

vertical_spacing = 0.42
ridge_height = 0.72
fig, ax = plt.subplots(figsize=(11.5, 7.6), constrained_layout=True)

for idx, row in enumerate(rows):
    y0 = idx * vertical_spacing
    data = row["pc1"]

    kde = stats.gaussian_kde(data)
    dens = kde(x_grid)
    dens = dens / (dens.max() + 1e-12)
    ridge = y0 + ridge_height * dens

    color = climate_palette[row["climate"]]
    ax.fill_between(x_grid, y0, ridge, color=color, alpha=0.45, linewidth=0)
    ax.plot(x_grid, ridge, color=color, linewidth=1.35, alpha=0.97)

    mu = float(np.mean(data))
    ql, qh = np.nanpercentile(data, [2.0, 98.0])
    ax.plot([mu, mu], [y0, y0 + 0.22], color="#1F2937", linewidth=1.0)
    ax.plot([ql, ql], [y0, y0 + 0.14], color="#4B5563", linewidth=0.85)
    ax.plot([qh, qh], [y0, y0 + 0.14], color="#4B5563", linewidth=0.85)

y_ticks = [i * vertical_spacing for i in range(len(rows))]
y_labels = [f"{r['setup']} | {r['climate']}" for r in rows]
ax.set_yticks(y_ticks)
ax.set_yticklabels(y_labels, fontsize=9)

ax.set_xlabel("PC1")
ax.set_ylabel("Setup | Climate")
ax.set_title("Compact ridgeline of PC1 distributions (historical vs ssp585)")
ax.grid(axis="x", alpha=0.22)
ax.grid(axis="y", alpha=0.05)

legend_handles = [
    Line2D([0], [0], color=climate_palette[c], linewidth=6, alpha=0.6, label=c)
    for c in target_climates
]
ax.legend(handles=legend_handles, title="Climate", loc="upper right")

plt.show()

In [ ]:
# Compact overlapped densities: PC1 for historical vs ssp585, one row per setup
target_climates = ["historical", "ssp585"]

if "latent_common_scores_by_setup_climate" not in globals():
    raise RuntimeError("latent_common_scores_by_setup_climate is missing. Run the setup/climate PCA cells first.")

setup_specs = [
    {"label": "AEh", "mode": "precomputed", "setup": "AEh"},
    {"label": "AEhs2", "mode": "precomputed", "setup": "AEhs2"},
    {"label": "AEhs2s3", "mode": "precomputed", "setup": "AEhs2s3"},
    {"label": "AEall", "mode": "precomputed", "setup": "AEall"},
    {"label": "exp4_aligned", "mode": "reproject", "source": "exp4_full", "slice_mode": "aligned48"},
    {"label": "exp4_non_aligned", "mode": "reproject", "source": "exp4_full", "slice_mode": "non_aligned16"},
    {"label": "exp5_cera_aligned", "mode": "reproject", "source": "exp5_cera_full", "slice_mode": "aligned48"},
    {"label": "exp5_cera_non_aligned", "mode": "reproject", "source": "exp5_cera_full", "slice_mode": "non_aligned16"},
    {"label": "exp5_baseline2_aligned", "mode": "reproject", "source": "exp5_baseline2_full", "slice_mode": "aligned48"},
    {"label": "exp5_baseline2_non_aligned", "mode": "reproject", "source": "exp5_baseline2_full", "slice_mode": "non_aligned16"},
    {"label": "exp5_cera_latent",    "mode": "reproject", "source": "exp5_cera_full_latent",    "slice_mode": "full"},
    {"label": "exp5_baseline_climax","mode": "reproject", "source": "exp5_baseline_climax_full","slice_mode": "full"}
]

rows = []
for spec in setup_specs:
    setup_label = spec["label"]
    if spec["mode"] == "precomputed":
        setup_block = latent_common_scores_by_setup_climate.get(spec["setup"], {})
        for climate in target_climates:
            scores = setup_block.get(climate) if isinstance(setup_block, dict) else None
            if scores is None:
                continue
            arr = np.asarray(scores)
            if arr.size == 0:
                continue
            pc1 = arr[:, 0] if arr.ndim > 1 else arr.astype(float)
            pc1 = pc1[np.isfinite(pc1)]
            if pc1.size < 10:
                continue
            rows.append({"setup": setup_label, "climate": climate, "pc1": pc1})
        continue

    if "latent_source_by_name" not in globals() or "_build_masked_input" not in globals() or "_project_in_common_pca" not in globals() or "_pad_sliced_input_for_pca" not in globals():
        raise RuntimeError("Missing latent projection helpers. Run the setup metric computation cell first.")

    latent_by_climate_source, _ = latent_source_by_name[spec["source"]]
    full_dim = latent_common_pca.n_features_in_
    
    for climate in target_climates:
        if climate not in latent_by_climate_source:
            continue
        X_raw = np.asarray(latent_by_climate_source[climate])
        if X_raw.size == 0:
            continue
        X_sliced = _build_masked_input(X_raw, spec["slice_mode"])
        X_proj_input = _pad_sliced_input_for_pca(X_sliced, spec["slice_mode"], full_dim)
        scores = _project_in_common_pca(X_proj_input)
        arr = np.asarray(scores)
        pc1 = arr[:, 0] if arr.ndim > 1 else arr.astype(float)
        pc1 = pc1[np.isfinite(pc1)]
        if pc1.size < 10:
            continue
        rows.append({"setup": setup_label, "climate": climate, "pc1": pc1})

if not rows:
    raise RuntimeError("No valid PC1 data found for the selected setups/climates.")

all_pc1 = np.concatenate([r["pc1"] for r in rows])
x_lo, x_hi = np.nanpercentile(all_pc1, [1.0, 99.0])
pad = 0.08 * (x_hi - x_lo + 1e-8)
x_grid = np.linspace(x_lo - pad, x_hi + pad, 450)

climate_palette = {
    "historical": climate_colors.get("historical", _base_palette[0]),
    "ssp585": climate_colors.get("ssp585", _base_palette[3]),
}

setup_order = [s["label"] for s in setup_specs]
rows = sorted(rows, key=lambda r: (setup_order.index(r["setup"]), 0 if r["climate"] == "historical" else 1))

fig, axes = plt.subplots(
    nrows=len(setup_order),
    ncols=1,
    figsize=(12.2, 1.55 * len(setup_order) + 1.2),
    sharex=True,
    constrained_layout=True,
 )

axes = np.atleast_1d(axes)

for ax, setup_label in zip(axes, setup_order):
    ax.set_title(setup_label, loc="left", fontsize=11, fontweight="semibold")
    for climate in target_climates:
        entry = next((r for r in rows if r["setup"] == setup_label and r["climate"] == climate), None)
        if entry is None:
            continue

        data = entry["pc1"]
        kde = stats.gaussian_kde(data)
        dens = kde(x_grid)
        dens = dens / (dens.max() + 1e-12)

        color = climate_palette[climate]
        alpha = 0.35 if climate == "historical" else 0.25
        ax.fill_between(x_grid, 0, dens, color=color, alpha=alpha, linewidth=0)
        ax.plot(x_grid, dens, color=color, linewidth=1.3)

        mu = float(np.mean(data))
        ql, qh = np.nanpercentile(data, [2.0, 98.0])
        ax.vlines(mu, 0, 1.02, color=color, linewidth=1.0, alpha=0.95)
        ax.vlines([ql, qh], 0, 0.16, color=color, linewidth=0.85, alpha=0.85, linestyle="--")

    ax.set_ylim(0, 1.08)
    ax.grid(axis="x", alpha=0.16)
    ax.grid(axis="y", alpha=0.06)
    ax.spines[["top", "right", "left"]].set_visible(False)
    ax.set_yticks([])

axes[-1].set_xlabel("PC1")
for ax in axes[:-1]:
    ax.tick_params(labelbottom=False)

fig.suptitle("Overlapped PC1 densities by setup: historical vs ssp585", y=1.01, fontsize=14, fontweight="semibold")
legend_handles = [
    Line2D([0], [0], color=climate_palette[c], linewidth=6, alpha=0.6, label=c)
    for c in target_climates
]
ax.legend(handles=legend_handles, loc="upper right", fontsize=10)

plt.show()

In [ ]:
# Compact overlapped densities: PC1 for 4 climates, one row per setup
target_climates = ["historical", "ssp245", "ssp370", "ssp585"]

if "latent_common_scores_by_setup_climate" not in globals():
    raise RuntimeError("latent_common_scores_by_setup_climate is missing. Run the setup/climate PCA cells first.")

setup_specs = [
    {"label": "AEh", "mode": "precomputed", "setup": "AEh"},
    {"label": "AEhs2", "mode": "precomputed", "setup": "AEhs2"},
    {"label": "AEhs2s3", "mode": "precomputed", "setup": "AEhs2s3"},
    {"label": "AEall", "mode": "precomputed", "setup": "AEall"},
    {"label": "exp4_aligned", "mode": "reproject", "source": "exp4_full", "slice_mode": "aligned48"},
    {"label": "exp4_non_aligned", "mode": "reproject", "source": "exp4_full", "slice_mode": "non_aligned16"},
    {"label": "exp5_cera_aligned", "mode": "reproject", "source": "exp5_cera_full", "slice_mode": "aligned48"},
    {"label": "exp5_cera_non_aligned", "mode": "reproject", "source": "exp5_cera_full", "slice_mode": "non_aligned16"},
    {"label": "exp5_baseline2_aligned", "mode": "reproject", "source": "exp5_baseline2_full", "slice_mode": "aligned48"},
    {"label": "exp5_baseline2_non_aligned", "mode": "reproject", "source": "exp5_baseline2_full", "slice_mode": "non_aligned16"},
    {"label": "exp5_cera_latent",    "mode": "reproject", "source": "exp5_cera_full_latent",    "slice_mode": "full"},
    {"label": "exp5_baseline_climax","mode": "reproject", "source": "exp5_baseline_climax_full","slice_mode": "full"}
]

rows = []
for spec in setup_specs:
    setup_label = spec["label"]
    if spec["mode"] == "precomputed":
        setup_block = latent_common_scores_by_setup_climate.get(spec["setup"], {})
        for climate in target_climates:
            scores = setup_block.get(climate) if isinstance(setup_block, dict) else None
            if scores is None:
                continue
            arr = np.asarray(scores)
            if arr.size == 0:
                continue
            pc1 = arr[:, 0] if arr.ndim > 1 else arr.astype(float)
            pc1 = pc1[np.isfinite(pc1)]
            if pc1.size < 10:
                continue
            rows.append({"setup": setup_label, "climate": climate, "pc1": pc1})
        continue

    if "latent_source_by_name" not in globals() or "_build_masked_input" not in globals() or "_project_in_common_pca" not in globals() or "_pad_sliced_input_for_pca" not in globals():
        raise RuntimeError("Missing latent projection helpers. Run the setup metric computation cell first.")

    latent_by_climate_source, _ = latent_source_by_name[spec["source"]]
    full_dim = latent_common_pca.n_features_in_
    
    for climate in target_climates:
        if climate not in latent_by_climate_source:
            continue
        X_raw = np.asarray(latent_by_climate_source[climate])
        if X_raw.size == 0:
            continue
        X_sliced = _build_masked_input(X_raw, spec["slice_mode"])
        X_proj_input = _pad_sliced_input_for_pca(X_sliced, spec["slice_mode"], full_dim)
        scores = _project_in_common_pca(X_proj_input)
        arr = np.asarray(scores)
        pc1 = arr[:, 0] if arr.ndim > 1 else arr.astype(float)
        pc1 = pc1[np.isfinite(pc1)]
        if pc1.size < 10:
            continue
        rows.append({"setup": setup_label, "climate": climate, "pc1": pc1})

if not rows:
    raise RuntimeError("No valid PC1 data found for the selected setups/climates.")

all_pc1 = np.concatenate([r["pc1"] for r in rows])
x_lo, x_hi = np.nanpercentile(all_pc1, [1.0, 99.0])
pad = 0.08 * (x_hi - x_lo + 1e-8)
x_grid = np.linspace(x_lo - pad, x_hi + pad, 450)

climate_palette = {
    "historical": climate_colors.get("historical", _base_palette[0]),
    "ssp245": climate_colors.get("ssp245", _base_palette[1]),
    "ssp370": climate_colors.get("ssp370", _base_palette[2]),
    "ssp585": climate_colors.get("ssp585", _base_palette[3]),
}

setup_order = [s["label"] for s in setup_specs]
rows = sorted(rows, key=lambda r: (setup_order.index(r["setup"]), target_climates.index(r["climate"])))

fig, axes = plt.subplots(
    nrows=len(setup_order),
    ncols=1,
    figsize=(12.2, 1.55 * len(setup_order) + 1.35),
    sharex=True,
    constrained_layout=True,
 )

axes = np.atleast_1d(axes)

for ax, setup_label in zip(axes, setup_order):
    ax.set_title(setup_label, loc="left", fontsize=11, fontweight="semibold")
    for climate in target_climates:
        entry = next((r for r in rows if r["setup"] == setup_label and r["climate"] == climate), None)
        if entry is None:
            continue

        data = entry["pc1"]
        kde = stats.gaussian_kde(data)
        dens = kde(x_grid)
        dens = dens / (dens.max() + 1e-12)

        color = climate_palette[climate]
        alpha = 0.23 if climate == "historical" else 0.18
        ax.fill_between(x_grid, 0, dens, color=color, alpha=alpha, linewidth=0)
        ax.plot(x_grid, dens, color=color, linewidth=1.2)

        mu = float(np.mean(data))
        ql, qh = np.nanpercentile(data, [2.0, 98.0])
        ax.vlines(mu, 0, 1.02, color=color, linewidth=1.0, alpha=0.95)
        ax.vlines([ql, qh], 0, 0.16, color=color, linewidth=0.8, alpha=0.85, linestyle="--")

    ax.set_ylim(0, 1.08)
    ax.grid(axis="x", alpha=0.16)
    ax.grid(axis="y", alpha=0.06)
    ax.spines[["top", "right", "left"]].set_visible(False)
    ax.set_yticks([])

axes[-1].set_xlabel("PC1")
for ax in axes[:-1]:
    ax.tick_params(labelbottom=False)

fig.suptitle("Overlapped PC1 densities by setup: historical, ssp245, ssp370, ssp585", y=1.01, fontsize=14, fontweight="semibold")
legend_handles = [
    Line2D([0], [0], color=climate_palette[c], linewidth=6, alpha=0.6, label=c)
    for c in target_climates
]
ax.legend(handles=legend_handles, loc="upper right", fontsize=10)

plt.show()

In [ ]:
# Heatmap: Wasserstein distance to historical by setup and SSP climate
target_climates = ["ssp245", "ssp370", "ssp585"]

if "latent_pairwise_metrics_all_setups_df" not in globals():
    raise RuntimeError("latent_pairwise_metrics_all_setups_df is missing. Run the latent metric cells first.")

# Use the actual setup keys stored in the pairwise metrics DataFrame.
setup_specs = [
    {"setup_key": "AEh", "display_name": "AEh"},
    {"setup_key": "AEhs2", "display_name": "AEhs2"},
    {"setup_key": "AEhs2s3", "display_name": "AEhs2s3"},
    {"setup_key": "AEall", "display_name": "AEall"},
    {"setup_key": "exp4_aligned48", "display_name": "exp4_aligned"},
    {"setup_key": "exp4_non_aligned16", "display_name": "exp4_non_aligned"},
    {"setup_key": "exp5_cera_aligned48", "display_name": "exp5_cera_aligned"},
    {"setup_key": "exp5_cera_non_aligned16", "display_name": "exp5_cera_non_aligned"},
    {"setup_key": "exp5_baseline2_full", "display_name": "exp5_baseline2_full"},
    {"setup_key": "exp5_cera_full_latent", "display_name": "exp5_cera_latent"},
    {"setup_key": "exp5_baseline_climax_full", "display_name": "exp5_baseline_climax"}
]

setup_order = [spec["display_name"] for spec in setup_specs]

rows = []
for spec in setup_specs:
    setup_key = spec["setup_key"]
    display_name = spec["display_name"]

    setup_rows = latent_pairwise_metrics_all_setups_df[
        (latent_pairwise_metrics_all_setups_df["setup"] == setup_key)
        & (latent_pairwise_metrics_all_setups_df["climate_i"] == "historical")
        & (latent_pairwise_metrics_all_setups_df["climate_j"].isin(target_climates))
    ].copy()

    if setup_rows.empty:
        continue

    for climate in target_climates:
        metric_row = setup_rows[setup_rows["climate_j"] == climate]
        if metric_row.empty:
            continue
        rows.append({
            "setup": display_name,
            "climate": climate,
            "wasserstein": float(metric_row["wasserstein_mean"].iloc[0]),
        })

if not rows:
    raise RuntimeError("No centroid distance rows were found for the requested heatmap.")

heatmap_df = (
    pd.DataFrame(rows)
    .pivot(index="climate", columns="setup", values="wasserstein")
    .reindex(index=target_climates, columns=setup_order)
)

fig_h = 5.8
fig, ax = plt.subplots(figsize=(max(12, 0.95 * len(setup_order) + 3.2), fig_h), constrained_layout=True)

values = heatmap_df.values
vmax = np.nanmax(values) if np.isfinite(values).any() else 1.0
im = ax.imshow(values, aspect="auto", cmap="Blues", vmin=0, vmax=vmax)

ax.set_xticks(range(len(heatmap_df.columns)))
ax.set_xticklabels(heatmap_df.columns, rotation=45, ha="right")
ax.set_yticks(range(len(heatmap_df.index)))
ax.set_yticklabels(heatmap_df.index)
ax.set_xlabel("Setup")
ax.set_ylabel("Climate")
ax.set_title("Wasserstein distance to historical by setup and climate")
ax.grid(False)

for i in range(heatmap_df.shape[0]):
    for j in range(heatmap_df.shape[1]):
        value = heatmap_df.values[i, j]
        text = "nan" if not np.isfinite(value) else f"{value:.9f}"
        ax.text(j, i, text, ha="center", va="center", fontsize=10, color="black", fontweight="semibold")

plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04, label="Wasserstein distance")

plt.show()

In [ ]:
# Heatmap: Wasserstein distance to historical by setup and SSP climate
target_climates = ["ssp245", "ssp370", "ssp585"]

if "latent_pairwise_metrics_all_setups_df" not in globals():
    raise RuntimeError("latent_pairwise_metrics_all_setups_df is missing. Run the latent metric cells first.")

# Use the actual setup keys stored in the pairwise metrics DataFrame.
setup_specs = [
    {"setup_key": "AEh", "display_name": "AEh"},
    {"setup_key": "AEhs2", "display_name": "AEhs2"},
    {"setup_key": "AEhs2s3", "display_name": "AEhs2s3"},
    {"setup_key": "AEall", "display_name": "AEall"},
    {"setup_key": "exp4_aligned48", "display_name": "exp4_aligned"},
    {"setup_key": "exp4_non_aligned16", "display_name": "exp4_non_aligned"},
    {"setup_key": "exp5_cera_aligned48", "display_name": "exp5_cera_aligned"},
    {"setup_key": "exp5_cera_non_aligned16", "display_name": "exp5_cera_non_aligned"},
    {"setup_key": "exp5_baseline2_full", "display_name": "exp5_baseline2_full"},
    {"setup_key": "exp5_cera_full_latent", "display_name": "exp5_cera_latent"},
    {"setup_key": "exp5_baseline_climax_full", "display_name": "exp5_baseline_climax"}
]

setup_order = [spec["display_name"] for spec in setup_specs]

rows = []
for spec in setup_specs:
    setup_key = spec["setup_key"]
    display_name = spec["display_name"]

    setup_rows = latent_pairwise_metrics_all_setups_df[
        (latent_pairwise_metrics_all_setups_df["setup"] == setup_key)
        & (latent_pairwise_metrics_all_setups_df["climate_i"] == "historical")
        & (latent_pairwise_metrics_all_setups_df["climate_j"].isin(target_climates))
    ].copy()

    if setup_rows.empty:
        continue

    for climate in target_climates:
        metric_row = setup_rows[setup_rows["climate_j"] == climate]
        if metric_row.empty:
            continue
        rows.append({
            "setup": display_name,
            "climate": climate,
            "wasserstein": np.sqrt(np.sqrt(1000000 * float(metric_row["wasserstein_mean"].iloc[0]))),
        })

if not rows:
    raise RuntimeError("No centroid distance rows were found for the requested heatmap.")

heatmap_df = (
    pd.DataFrame(rows)
    .pivot(index="climate", columns="setup", values="wasserstein")
    .reindex(index=target_climates, columns=setup_order)
)

fig_h = 5.8
fig, ax = plt.subplots(figsize=(max(12, 0.95 * len(setup_order) + 3.2), fig_h), constrained_layout=True)

values = heatmap_df.values
vmax = np.nanmax(values) if np.isfinite(values).any() else 1.0
im = ax.imshow(values, aspect="auto", cmap="Blues", vmin=0, vmax=vmax)

ax.set_xticks(range(len(heatmap_df.columns)))
ax.set_xticklabels(heatmap_df.columns, rotation=45, ha="right")
ax.set_yticks(range(len(heatmap_df.index)))
ax.set_yticklabels(heatmap_df.index)
ax.set_xlabel("Setup")
ax.set_ylabel("Climate")
ax.set_title("Wasserstein distance to historical by setup and climate sqrt(sqrt(1000000 * distance))", fontsize=11)
ax.grid(False)

for i in range(heatmap_df.shape[0]):
    for j in range(heatmap_df.shape[1]):
        value = heatmap_df.values[i, j]
        text = "nan" if not np.isfinite(value) else f"{value:.9f}"
        ax.text(j, i, text, ha="center", va="center", fontsize=10, color="black", fontweight="semibold")

plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04, label="Wasserstein distance")

plt.show()

In [ ]:
# Rotated ridgeline: PC1 densities by setup, with setup on the x-axis and density extending sideways
target_climates = ["historical", "ssp245", "ssp370", "ssp585"]

if "latent_common_scores_by_setup_climate" not in globals():
    raise RuntimeError("latent_common_scores_by_setup_climate is missing. Run the setup/climate PCA cells first.")

setup_specs = [
    {"raw_name": "AEh", "display_name": "AEh", "mode": "precomputed"},
    {"raw_name": "AEhs2", "display_name": "AEhs2", "mode": "precomputed"},
    {"raw_name": "AEhs2s3", "display_name": "AEhs2s3", "mode": "precomputed"},
    {"raw_name": "AEall", "display_name": "AEall", "mode": "precomputed"},
    {"raw_name": "exp4_full", "display_name": "exp4_aligned", "mode": "reproject", "slice_mode": "aligned48"},
    {"raw_name": "exp4_full", "display_name": "exp4_non_aligned", "mode": "reproject", "slice_mode": "non_aligned16"},
    {"raw_name": "exp5_cera_full", "display_name": "exp5_cera_aligned", "mode": "reproject", "slice_mode": "aligned48"},
    {"raw_name": "exp5_cera_full", "display_name": "exp5_cera_non_aligned", "mode": "reproject", "slice_mode": "non_aligned16"},
    {"raw_name": "exp5_baseline2_full", "display_name": "exp5_baseline2_aligned", "mode": "reproject", "slice_mode": "aligned48"},
    {"raw_name": "exp5_baseline2_full", "display_name": "exp5_baseline2_non_aligned", "mode": "reproject", "slice_mode": "non_aligned16"},
    {"raw_name": "exp5_cera_full_latent", "display_name": "exp5_cera_latent", "mode": "reproject", "slice_mode": "full"},
    {"raw_name": "exp5_baseline_climax_full", "display_name": "exp5_baseline_climax", "mode": "reproject", "slice_mode": "full"}
]

setup_order = [spec["display_name"] for spec in setup_specs]
climate_rank = {c: i for i, c in enumerate(target_climates)}

def _get_setup_climate_scores(spec, climate):
    if spec["mode"] == "precomputed":
        setup_block = latent_common_scores_by_setup_climate.get(spec["raw_name"], {})
        scores = setup_block.get(climate) if isinstance(setup_block, dict) else None
        return None if scores is None else np.asarray(scores)

    if "latent_source_by_name" not in globals() or "_build_masked_input" not in globals() or "_project_in_common_pca" not in globals() or "_pad_sliced_input_for_pca" not in globals():
        raise RuntimeError("Missing latent projection helpers. Run the setup metric computation cell first.")

    latent_by_climate_source, _ = latent_source_by_name[spec["raw_name"]]
    X_raw = latent_by_climate_source.get(climate)
    if X_raw is None:
        return None
    X_raw = np.asarray(X_raw)
    if X_raw.size == 0:
        return None
    full_dim = latent_common_pca.n_features_in_
    X_sliced = _build_masked_input(X_raw, spec["slice_mode"])
    X_proj_input = _pad_sliced_input_for_pca(X_sliced, spec["slice_mode"], full_dim)
    return _project_in_common_pca(X_proj_input)

rows = []
for spec in setup_specs:
    for climate in target_climates:
        scores = _get_setup_climate_scores(spec, climate)
        if scores is None or np.asarray(scores).size == 0:
            continue
        arr = np.asarray(scores)
        pc1 = arr[:, 0] if arr.ndim > 1 else arr.astype(float)
        pc1 = pc1[np.isfinite(pc1)]
        if pc1.size < 10:
            continue
        rows.append({"setup": spec["display_name"], "climate": climate, "pc1": pc1})

if not rows:
    raise RuntimeError("No valid PC1 data found for the selected setups/climates.")

all_pc1 = np.concatenate([r["pc1"] for r in rows])
y_lo, y_hi = np.nanpercentile(all_pc1, [1.0, 99.0])
pad = 0.08 * (y_hi - y_lo + 1e-8)
y_grid = np.linspace(y_lo - pad, y_hi + pad, 420)

climate_palette = {
    "historical": climate_colors.get("historical", _base_palette[0]),
    "ssp245": climate_colors.get("ssp245", _base_palette[1]),
    "ssp370": climate_colors.get("ssp370", _base_palette[2]),
    "ssp585": climate_colors.get("ssp585", _base_palette[3]),
}

setup_spacing = 1.2
climate_offsets = {
    "historical": -0.30,
    "ssp245": -0.10,
    "ssp370": 0.10,
    "ssp585": 0.30,
}

fig, ax = plt.subplots(figsize=(16, 7.2), constrained_layout=True)

for setup_index, setup_label in enumerate(setup_order):
    x0 = setup_index * setup_spacing
    for climate in target_climates:
        entry = next((r for r in rows if r["setup"] == setup_label and r["climate"] == climate), None)
        if entry is None:
            continue

        data = entry["pc1"]
        kde = stats.gaussian_kde(data)
        density = kde(y_grid)
        density = density / (density.max() + 1e-12)

        x_center = x0 + climate_offsets[climate]
        width = 0.22 if climate == "historical" else 0.18
        color = climate_palette[climate]

        ax.fill_betweenx(y_grid, x_center, x_center + width * density, color=color, alpha=0.28, linewidth=0)
        ax.plot(x_center + width * density, y_grid, color=color, linewidth=1.25)
        ax.plot([x_center, x_center], [np.mean(data), np.mean(data)], marker="|", markersize=14, color=color, linewidth=0)

        ql, qh = np.nanpercentile(data, [2.0, 98.0])
        ax.plot([x_center + 0.02, x_center + width * 0.55], [np.mean(data), np.mean(data)], color=color, linewidth=1.0)
        ax.plot([x_center + 0.01, x_center + width * 0.42], [ql, ql], color=color, linewidth=0.9, linestyle="--")
        ax.plot([x_center + 0.01, x_center + width * 0.42], [qh, qh], color=color, linewidth=0.9, linestyle="--")

ax.set_xlim(-0.8, (len(setup_order) - 1) * setup_spacing + 0.9)
ax.set_xticks([i * setup_spacing for i in range(len(setup_order))])
ax.set_xticklabels(setup_order, rotation=45, ha="right")
ax.set_ylabel("PC1")
ax.set_xlabel("Setup")
ax.set_title("Rotated PC1 ridgeline by setup and climate")
ax.grid(axis="y", alpha=0.16)
ax.grid(axis="x", alpha=0.06)

legend_handles = [
    Line2D([0], [0], color=climate_palette[c], linewidth=6, alpha=0.6, label=c)
    for c in target_climates
]
ax.legend(handles=legend_handles, title="Climate", loc="upper right")

plt.show()

In [ ]:
# Q95 of multivariate norms by setup and climate
target_climates = ["historical", "ssp245", "ssp370", "ssp585"]

if "latent_common_scores_by_setup_climate" not in globals():
    raise RuntimeError("latent_common_scores_by_setup_climate is missing. Run the setup/climate PCA cells first.")

setup_specs = [
    {"raw_name": "AEh", "display_name": "AEh", "mode": "precomputed"},
    {"raw_name": "AEhs2", "display_name": "AEhs2", "mode": "precomputed"},
    {"raw_name": "AEhs2s3", "display_name": "AEhs2s3", "mode": "precomputed"},
    {"raw_name": "AEall", "display_name": "AEall", "mode": "precomputed"},
    {"raw_name": "exp4_full", "display_name": "exp4_aligned", "mode": "reproject", "slice_mode": "aligned48"},
    {"raw_name": "exp4_full", "display_name": "exp4_non_aligned", "mode": "reproject", "slice_mode": "non_aligned16"},
]



setup_order = [spec["display_name"] for spec in setup_specs]

def _get_scores_for_setup_climate(spec, climate):
    if spec["mode"] == "precomputed":
        setup_block = latent_common_scores_by_setup_climate.get(spec["raw_name"], {})
        scores = setup_block.get(climate) if isinstance(setup_block, dict) else None
        return None if scores is None else np.asarray(scores)

    if "latent_source_by_name" not in globals() or "_build_masked_input" not in globals() or "_project_in_common_pca" not in globals() or "_pad_sliced_input_for_pca" not in globals():
        raise RuntimeError("Missing latent projection helpers. Run the setup metric computation cell first.")

    latent_by_climate_source, _ = latent_source_by_name[spec["raw_name"]]
    X_raw = latent_by_climate_source.get(climate)
    if X_raw is None:
        return None
    X_raw = np.asarray(X_raw)
    if X_raw.size == 0:
        return None
    full_dim = latent_common_pca.n_features_in_
    X_sliced = _build_masked_input(X_raw, spec["slice_mode"])
    X_proj_input = _pad_sliced_input_for_pca(X_sliced, spec["slice_mode"], full_dim)
    return _project_in_common_pca(X_proj_input)

rows = []
for spec in setup_specs:
    for climate in target_climates:
        scores = _get_scores_for_setup_climate(spec, climate)
        if scores is None or np.asarray(scores).size == 0:
            continue
        norm = np.linalg.norm(np.asarray(scores), axis=1)
        q95 = float(np.quantile(norm, 0.95))
        rows.append({"setup": spec["display_name"], "climate": climate, "q95_norm": q95})

if not rows:
    raise RuntimeError("No Q95 rows were found for the requested chart.")

q95_df = (
    pd.DataFrame(rows)
    .pivot(index="setup", columns="climate", values="q95_norm")
    .reindex(index=setup_order, columns=target_climates)
)

fig, ax = plt.subplots(figsize=(16, 4.8), constrained_layout=True)

climate_palette = {
    "historical": climate_colors.get("historical", _base_palette[0]),
    "ssp245": climate_colors.get("ssp245", _base_palette[1]),
    "ssp370": climate_colors.get("ssp370", _base_palette[2]),
    "ssp585": climate_colors.get("ssp585", _base_palette[3]),
}

x = np.arange(len(q95_df.index))
for climate in target_climates:
    ax.plot(
        x,
        q95_df[climate].values,
        color=climate_palette[climate],
        marker="",
        markersize=4,
        linewidth=1.2,
        alpha=0.95,
        label=climate,
    )

ax.set_xticks(x)
ax.set_xticklabels(q95_df.index, rotation=45, ha="right")
ax.set_xlabel("Setup")
ax.set_ylabel("Q95 of ||PCA score||")
ax.set_title("95th percentile of multivariate distribution norms by setup and climate")
ax.grid(axis="y", alpha=0.22)
ax.legend(title="Climate", ncol=2, loc="upper right")

plt.show()

Centered Kernel Alignment (CKA) distance graph (without PCA)

In [ ]:
# CKA between historical and SSP latent representations by setup
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

if "latent_source_by_name" not in globals() or "latent_common_scores_by_setup_climate" not in globals():
    raise RuntimeError("Missing latent scores. Run the latent setup cells first.")

# Keep the same setup order used in part 3 plots.
setup_specs = [
    {"display_name": "AEh", "mode": "precomputed", "source_name": "AEh", "slice_mode": "full"},
    {"display_name": "AEhs2", "mode": "precomputed", "source_name": "AEhs2", "slice_mode": "full"},
    {"display_name": "AEhs2s3", "mode": "precomputed", "source_name": "AEhs2s3", "slice_mode": "full"},
    {"display_name": "AEall", "mode": "precomputed", "source_name": "AEall", "slice_mode": "full"},
    {"display_name": "exp4_aligned48", "mode": "reproject", "source_name": "exp4_full", "slice_mode": "aligned48"},
    {"display_name": "exp4_non_aligned16", "mode": "reproject", "source_name": "exp4_full", "slice_mode": "non_aligned16"},
    {"display_name": "exp5_cera_aligned48", "mode": "reproject", "source_name": "exp5_cera_full", "slice_mode": "aligned48"},
    {"display_name": "exp5_cera_non_aligned16", "mode": "reproject", "source_name": "exp5_cera_full", "slice_mode": "non_aligned16"},
    {"display_name": "exp5_baseline2_aligned48", "mode": "reproject", "source_name": "exp5_baseline2_full", "slice_mode": "aligned48"},
    {"display_name": "exp5_baseline2_non_aligned16", "mode": "reproject", "source_name": "exp5_baseline2_full", "slice_mode": "non_aligned16"},
    {"display_name": "exp5_cera_full_latent_in_training", "mode": "reproject", "source_name": "exp5_cera_full_latent", "slice_mode": "full"},
    {"display_name": "exp5_baseline_climax", "mode": "reproject", "source_name": "exp5_baseline_climax_full", "slice_mode": "full"}
]

target_climates = ["ssp245", "ssp370", "ssp585"]
setup_order = [spec["display_name"] for spec in setup_specs]

if "climate_colors" not in globals():
    raise RuntimeError("climate_colors is missing. Run the plotting setup cells first.")


def _center_and_normalize(X):
    """Center columns and normalize the block per-setup."""
    X = np.asarray(X, dtype=np.float64)
    # center columns (features)
    Xc = X - np.mean(X, axis=0, keepdims=True)
    # normalize by Frobenius norm to give comparable scale across setups
    fro = np.linalg.norm(Xc, ord='fro')
    if fro <= 0:
        return Xc
    return Xc / fro


def _linear_cka(X, Y):
    """Compute linear CKA after centering+normalizing each block."""
    Xc = _center_and_normalize(X)
    Yc = _center_and_normalize(Y)
    xty = Xc.T @ Yc
    xxt = Xc.T @ Xc
    yyt = Yc.T @ Yc
    hsic_xy = float(np.sum(xty * xty))
    hsic_xx = float(np.sum(xxt * xxt))
    hsic_yy = float(np.sum(yyt * yyt))
    denom = np.sqrt(max(hsic_xx, 1e-12) * max(hsic_yy, 1e-12))
    return float(hsic_xy / denom) if denom > 0 else np.nan


def _get_metadata_for_setup_climate(spec, climate):
    """Retrieve metadata for latent samples (if available)."""
    if "latent_source_by_name" not in globals():
        return None
    
    source_name = spec["source_name"]
    if source_name not in latent_source_by_name:
        return None
    
    _, metadata_by_climate_source = latent_source_by_name[source_name]
    return metadata_by_climate_source.get(climate) if metadata_by_climate_source else None


def _get_scores_for_setup_climate(spec, climate):
    # Return raw latent representations (sliced if requested).
    if "latent_source_by_name" not in globals():
        raise RuntimeError("Missing latent sources. Run the latent setup cells first.")

    latent_by_climate_source, _ = latent_source_by_name[spec["source_name"]]
    X_raw = latent_by_climate_source.get(climate)
    if X_raw is None:
        return None
    X_raw = np.asarray(X_raw)
    if X_raw.size == 0:
        return None

    # If a slice mode is requested, return the sliced raw latent dims (no PCA).
    slice_mode = spec.get("slice_mode", "full")
    if slice_mode == "full":
        return X_raw
    if "_build_masked_input" not in globals():
        raise RuntimeError("_build_masked_input is required for sliced latent retrieval.")
    return np.asarray(_build_masked_input(X_raw, slice_mode))


def _pair_samples_by_metadata(hist_metadata, ssp_metadata, hist_scores, ssp_scores):
    """
    Pair samples between historical and SSP based on common metadata keys.
    Tries to match by model/source identifiers in metadata.
    
    Returns:
        (hist_block, ssp_block): paired score arrays of same length
    """
    if hist_metadata is None or ssp_metadata is None:
        # If no metadata available, use stratified sampling: take same fraction from each
        sample_size = min(len(hist_scores), len(ssp_scores))
        return hist_scores[:sample_size], ssp_scores[:sample_size]
    
    hist_df = pd.DataFrame(hist_metadata) if isinstance(hist_metadata, list) else hist_metadata
    ssp_df = pd.DataFrame(ssp_metadata) if isinstance(ssp_metadata, list) else ssp_metadata
    
    # Try to find common keys to match on (e.g., 'model', 'source', 'ensemble')
    common_cols = set(hist_df.columns) & set(ssp_df.columns)
    if not common_cols:
        # No common metadata columns - use same-index pairing
        sample_size = min(len(hist_scores), len(ssp_scores))
        return hist_scores[:sample_size], ssp_scores[:sample_size]
    
    # Merge on common metadata columns
    merged = hist_df.reset_index(drop=False).merge(
        ssp_df.reset_index(drop=False),
        on=list(common_cols),
        how='inner',
        suffixes=('_hist', '_ssp')
    )
    
    if len(merged) == 0:
        # Merge failed - fall back to same-index pairing
        sample_size = min(len(hist_scores), len(ssp_scores))
        return hist_scores[:sample_size], ssp_scores[:sample_size]
    
    hist_indices = merged['index_hist'].values.astype(int)
    ssp_indices = merged['index_ssp'].values.astype(int)
    
    return hist_scores[hist_indices], ssp_scores[ssp_indices]


sample_size_cap = 4096
rows = []

for spec in setup_specs:
    hist_scores = _get_scores_for_setup_climate(spec, "historical")
    if hist_scores is None or np.asarray(hist_scores).size == 0:
        continue

    hist_scores = np.asarray(hist_scores)
    hist_metadata = _get_metadata_for_setup_climate(spec, "historical")
    
    for climate in target_climates:
        ssp_scores = _get_scores_for_setup_climate(spec, climate)
        if ssp_scores is None or np.asarray(ssp_scores).size == 0:
            continue

        ssp_scores = np.asarray(ssp_scores)
        ssp_metadata = _get_metadata_for_setup_climate(spec, climate)
        
        # Pair samples correctly using metadata
        hist_block, ssp_block = _pair_samples_by_metadata(hist_metadata, ssp_metadata, hist_scores, ssp_scores)
        
        # Cap sample size
        sample_size = min(len(hist_block), len(ssp_block), sample_size_cap)
        if sample_size < 2:
            continue
        
        # Take capped samples
        hist_block = hist_block[:sample_size]
        ssp_block = ssp_block[:sample_size]

        rows.append({
            "setup": spec["display_name"],
            "climate": climate,
            "cka": float(_linear_cka(hist_block, ssp_block)),
            "n": int(sample_size),
        })

if not rows:
    raise RuntimeError("No CKA values could be computed for the selected setups and climates.")

cka_df = pd.DataFrame(rows)
cka_plot_df = cka_df.pivot(index="setup", columns="climate", values="cka").reindex(index=setup_order, columns=target_climates)

fig, ax = plt.subplots(figsize=(15.5, 5.8), constrained_layout=True)
climate_palette = {
    "historical": climate_colors.get("historical", "#2F3B52"),
    "ssp245": climate_colors.get("ssp245", "#2C7FB8"),
    "ssp370": climate_colors.get("ssp370", "#F28E2B"),
    "ssp585": climate_colors.get("ssp585", "#C0392B"),
}

x = np.arange(len(cka_plot_df.index))
for climate in target_climates:
    y = cka_plot_df[climate].values
    ax.plot(
        x,
        y,
        color=climate_palette[climate],
        linewidth=2.0,
        marker="o",
        markersize=5,
        label=climate,
    )

ax.set_xticks(x)
ax.set_xticklabels(cka_plot_df.index, rotation=45, ha="right")
ax.set_ylim(0.0, 0.01)
ax.set_ylabel("Linear CKA")
ax.set_xlabel("Setup")
ax.set_title("Linear CKA between historical and SSP latent representations by setup\n(with proper sample pairing)")
ax.grid(axis="y", alpha=0.22)
ax.legend(title="Climate", ncols=3, frameon=False, loc="upper left")

plt.show()